In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = 160
# to save model

arch_model = EfficientNetB0(
    weights=None,
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = arch_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

test_model = Model(inputs=arch_model.input, outputs=output)

test_model.load_weights("deepfake_weights.h5")

test_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## test data
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_data = test_gen.flow_from_directory(
    "Dataset/Test",
    target_size=(160,160),
    batch_size=16,
    class_mode='binary'
)

loss, acc = test_model.evaluate(test_data)
print("Test Accuracy:", acc)

from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "deepfake.jpg"

# Load image
img = image.load_img(img_path, target_size=(160, 160))

# Convert to array
img_array = image.img_to_array(img)

# Expand dimensions (VERY IMPORTANT)
img_array = np.expand_dims(img_array, axis=0)

# Preprocess (VERY IMPORTANT)
img_array = preprocess_input(img_array)

test_model.predict(img_array)    ## fake=0, real =1
